# SplitFed Demonstration Notebook

This notebook provides a simple interactive setup of the Split Federated Learning (SplitFed) simulator.
We will load the MNIST dataset, partition it among 3 simulated clients, and run a short training loop.

### Step 0: Remote Server Setup

If you are running this notebook on a remote Jupyter server (like Google Colab), run the following cell to clone the repository, install dependencies, and set the correct working directory.

In [6]:
import os, subprocess, sys
from pathlib import Path

repo_path = Path(r"C:\Users\BIM\ad-sfl")
repo_url  = "https://github.com/tomal66/ad-sfl.git"

def run(cmd, cwd=None):
    return subprocess.run(cmd, cwd=cwd, check=True, text=True, capture_output=True)

# 1) Clone if missing
if not repo_path.exists():
    repo_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", repo_url, str(repo_path)], check=True)

# 2) Verify it's a git repo
if not (repo_path / ".git").exists():
    raise RuntimeError(f"{repo_path} exists but is not a git repo (no .git). Fix folder nesting or set repo_path correctly.")

# 3) Fetch latest from origin
subprocess.run(["git", "fetch", "--all", "--prune"], cwd=repo_path, check=True)

# 4) Detect remote default branch (origin/HEAD -> origin/main, etc.)
default_ref = run(["git", "symbolic-ref", "refs/remotes/origin/HEAD"], cwd=repo_path).stdout.strip()
# default_ref looks like: refs/remotes/origin/main
branch = default_ref.split("/")[-1]

# 5) Align working tree exactly to latest remote commit on default branch
subprocess.run(["git", "checkout", branch], cwd=repo_path, check=True)
subprocess.run(["git", "reset", "--hard", f"origin/{branch}"], cwd=repo_path, check=True)

# Optional but often useful: remove untracked files/folders so you are *exactly* aligned
subprocess.run(["git", "clean", "-fd"], cwd=repo_path, check=True)

# 6) chdir + install requirements
os.chdir(repo_path)
req = repo_path / "requirements.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(req)], check=True)

# Print current commit info
commit = run(["git", "rev-parse", "HEAD"], cwd=repo_path).stdout.strip()
print("Repo:", repo_path)
print("Branch:", branch)
print("HEAD:", commit)
print("Current working directory:", os.getcwd())

Repo: C:\Users\BIM\ad-sfl
Branch: main
HEAD: aa185dd4c851a5354b7b781bc23f333cbc23f673
Current working directory: C:\Users\BIM\ad-sfl


In [7]:
import torch
from src.data.datasets import get_datasets
from src.data.partition import partition_data_iid
from src.models.split import ClientModel, ServerModel
from src.core.client import SplitFedClient
from src.core.server import SplitFedServer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


### Step 1: Load and Partition Data

In [8]:
num_clients = 3
batch_size = 64

print("Loading MNIST dataset...")
train_data, test_data = get_datasets("MNIST")

print("Partitioning data info IID subsets...")
client_datasets = partition_data_iid(train_data, num_clients)
print(f"Data partitioned to {len(client_datasets)} clients.")

Loading MNIST dataset...


100%|█████████████████████████████████████| 9.91M/9.91M [00:03<00:00, 3.18MB/s]
100%|██████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 117kB/s]
100%|██████████████████████████████████████| 1.65M/1.65M [00:01<00:00, 900kB/s]
100%|█████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 9.29MB/s]

Partitioning data info IID subsets...
Data partitioned to 3 clients.


### Step 2: Initialize Server and Client Models

In SplitFed, a portion of the network is on the client, and the rest is on the server.

In [9]:
import copy

# Server Model initialization
server_model = ServerModel(in_channels=32, hidden_channels=64, num_classes=10, input_size=(14, 14))
server = SplitFedServer(model=server_model, num_clients=num_clients, device=device)

# Client Models initialization (each client starts with same weights)
base_client_model = ClientModel(in_channels=1, hidden_channels=32)
clients = []
for i in range(num_clients):
    client_model = copy.deepcopy(base_client_model)
    client = SplitFedClient(client_id=i, model=client_model, dataset=client_datasets[i], batch_size=batch_size, device=device)
    clients.append(client)

print("Initialized 1 Server and 3 Clients.")

Initialized 1 Server and 3 Clients.


### Step 3: Simulation Loop

In [ ]:
import matplotlib.pyplot as plt
from tqdm.auto import trange
from src.algorithms import run_sfl_round

epochs = 100
historical_acc = []

pbar = trange(epochs, desc="Training", unit="epoch")
for epoch in pbar:
    avg_loss, avg_acc = run_sfl_round(clients, server)
    historical_acc.append(avg_acc)
    pbar.set_postfix(loss=f"{avg_loss:.4f}", acc=f"{avg_acc:.4f}")

print("Training simulation complete. Plotting accuracy...")
plt.figure(figsize=(8, 4))
plt.plot(range(1, epochs + 1), historical_acc, marker="o", markersize=3)
plt.title("SplitFed Training Accuracy over Rounds")
plt.xlabel("Communication Round / Epoch")
plt.ylabel("Accuracy")
plt.grid(True)
plt.show()
